# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}\n\nDescription: {metadata.description}")

# If you wish to see all metadata attributes
# print(metadata.to_json())

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below we enumerate all RecordSets (data tables) within the dataset, and display their `@id`, name, and the fields/columns they provide, also by `@id`.

All record sets and fields are referenced by their Croissant `@id` for clarity and reproducible analysis.

In [ ]:
# List and describe all RecordSets, Fields, and their IDs
record_sets = list(dataset.metadata.record_sets)

print(f"Number of record sets: {len(record_sets)}\n")
for rec in record_sets:
    print(f"RecordSet name: {rec.name}\n  @id: {rec.id}\n  Description: {rec.description}")
    print(f"  Fields and columns:")
    for field in rec.fields:
        print(f"    field @id: {field.id:45} label: {field.name}")
        if hasattr(field, 'column') and field.column is not None:
            if hasattr(field.column, 'id'):
                print(f"      column @id: {field.column.id}")
    print('-' * 60)

# For reference below, store main record_set @id(s)
all_record_set_ids = [rec.id for rec in record_sets]
all_field_ids = {rec.id: [f.id for f in rec.fields] for rec in record_sets}

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Use the variable `all_record_set_ids` to list available record sets and their corresponding field mappings.

In [ ]:
# Extract records for all record sets into DataFrames
dataframes = {}
for rec_id in all_record_set_ids:
    print(f"Loading records for record set {rec_id} ...")
    records = list(dataset.records(record_set=rec_id))
    if len(records) == 0:
        print(f"No records found for {rec_id}!")
    else:
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"DataFrame with columns: {df.columns.tolist()}")
        print(df.head())
    print('-'*40)

# If record sets are present, pick the first for demonstration:
if len(all_record_set_ids) > 0:
    main_record_set_id = all_record_set_ids[0]
    print(f"\nMain record set (@id) selected for downstream analysis: {main_record_set_id}")
    if main_record_set_id in dataframes:
        print(f"Column names: {dataframes[main_record_set_id].columns.tolist()}")
        display_cols = dataframes[main_record_set_id].columns.tolist()
        dataframes[main_record_set_id].head()
    else:
        print(f"No DataFrame found for {main_record_set_id}!")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

All entity references use Croissant `@id`

> **If you have a particular field you'd like to explore, use its `@id` and column name as identified in Section 2 & 3.**

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
# For demonstration, we select a numeric field for the main record set (replace with correct `@id`/column)
numeric_field_id = None
group_field_id = None
main_df = None
if len(all_record_set_ids) > 0 and all_record_set_ids[0] in dataframes:
    main_record_set_id = all_record_set_ids[0]
    main_df = dataframes[main_record_set_id]
    # Try to auto-detect a numeric field
    num_cols = main_df.select_dtypes(include=np.number).columns.tolist()
    if len(num_cols) > 0:
        numeric_field_id = num_cols[0]
        print(f"Selected numeric field/column: {numeric_field_id}")
    else:
        print("No numeric fields found. Please consult the field overview.")
    # Detect a potential group-by field
    non_num_cols = [col for col in main_df.columns if col not in num_cols]
    if len(non_num_cols) > 0:
        group_field_id = non_num_cols[0]

if main_df is not None and numeric_field_id is not None:
    threshold = main_df[numeric_field_id].mean()  # Use the mean for demo, or set to 10
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
    print(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field_id is not None and group_field_id in main_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("Not enough information for EDA - please review data extraction above to identify numeric fields.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below we plot the distribution of the main numeric field selected in EDA, and if possible, the aggregated mean per group.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Grouped bar plot if grouping is available
    if group_field_id is not None and group_field_id in main_df.columns:
        group_means = main_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)[:12]  # Show top 12
        plt.figure(figsize=(10,5))
        sns.barplot(x=group_means.index, y=group_means.values)
        plt.xticks(rotation=45)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print("Not enough data for visualization. Please check data extraction and EDA steps.")

## 6. Conclusion
In this notebook, we demonstrated loading, exploring, and analyzing the [FAIR^2 Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using `mlcroissant`. All data entities were referenced using their stable Croissant `@id` identifiers.

Key workflow steps included:
- Reading dataset metadata and structure.
- Exploring all available record sets and fields with their `@id`s.
- Loading tabular data for selected record sets into pandas DataFrames.
- Filtering records, normalizing numeric fields, and grouping records for analysis.
- Visualizing distributions and aggregated group statistics.

Further work can include analysis of other record sets, integration with experiment metadata, or more advanced machine learning workflows. For extended documentation, see the [`mlcroissant` documentation](https://github.com/mlcommons/croissant).